In [ ]:
# Import necessary libraries 
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns
import warnings

# Import scikit-learn metrics module for model evaluation
from sklearn import metrics

# Import TensorFlow and Keras for building and training neural networks
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense 
from tensorflow.keras.layers import Dropout, BatchNormalization
from keras.callbacks import EarlyStopping

# Set display options for pandas and suppress warnings
pd.set_option('display.max_columns', 100)
warnings.filterwarnings('ignore')

### *About Dataset*

##### * *This dataset encapsulates information about employee status, specifically highlighting terminations. For each of the 10 years covered, the dataset provides details on both active employees and those who have undergone termination.* 
##### * *The objective is to assess whether an employee is currently active or has been terminated based on the available data.*

In [ ]:
# Read the dataset from a CSV file into a Pandas DataFrame

data = pd.read_csv('MFG10YearTerminationData.csv')

In [ ]:
# Create a copy of the original DataFrame for further processing

df = data.copy()

In [ ]:
# Display information about the DataFrame, including data types and non-null counts

df.info()

In [ ]:
df.head()

### **Insights**

##### - *The dataset doesn't contain any null values, ensuring data completeness*
##### - *The dataset comprises a mix of integer and string datatype features*
##### - *Some columns are redundant and can be safely removed without affecting the further analysis*
##### - *Certain columns, like 'termreason_desc and termtype_desc directly reveal information our deep learning Model is attempting to predict, To prevent potential data leakage and maintain model integrity, it is recommended to exclude these columns from the feature set* 

In [ ]:
# Dropping unnecessary columns from the DataFrame to prepare the data for modeling.

df.drop(columns=['EmployeeID', 'gender_short', 'birthdate_key', 'terminationdate_key', 'length_of_service', 'termreason_desc', 'termtype_desc'], inplace=True)

In [ ]:
df.head()

In [ ]:
# Displaying the value counts for each column in the DataFrame to understand the distribution of categorical data.

for col in df.columns:
    counts = df[col].value_counts()
    print(f'{counts}\n')
    print('---------------------------')

In [ ]:
df.head()

### **EDA**

In [ ]:
# Displaying the count of unique values in the 'STATUS' column to understand the distribution of employee status.

df['STATUS'].value_counts()

In [ ]:
# Visualizing Employee Attrition Rate using a Pie Chart
plt.figure(figsize=(4,4))

plt.pie(df['STATUS'].value_counts(), colors=['#caf0f8', '#ffb3c6'], labels=['Active', 'Terminated'], autopct='%1.1f%%', explode=(0.1, 0.1))
plt.title('Employee Attrition Distribution', fontdict={'fontsize' : 10, 'weight' : 'bold', 'color' : '#6c757d'})

plt.show()

In [ ]:
terminated_df = df[df['STATUS'] == 'TERMINATED']

In [ ]:
# Creating a histogram to visualize the distribution of terminated employees Based on Age.
plt.figure(figsize=(8,3))

sns.histplot(x='age', data=terminated_df, bins=50, color='#023e8a', kde=True)
plt.title('Attrition Count Based on Age', fontdict={'fontsize' : 9, 'weight' : 'bold'})
plt.xlabel('Age', fontdict={'fontsize' : 8, 'weight' : 'bold'})
plt.ylabel('Frequency', fontdict={'fontsize' : 8, 'weight' : 'bold'})
sns.despine()
plt.show()

In [ ]:
terminated_df['gender_full'].value_counts()

In [ ]:
# Creating a Count Plot to visualize the distribution of terminated employees Based on Gender.
plt.figure(figsize=(5,4))

sns.countplot(x='gender_full', data=terminated_df, palette=['#48cae4', '#ffb3c6'])
plt.title('Attrition Count Based on Gender', fontdict={'fontsize' : 9, 'weight' : 'bold'})
plt.xlabel('Gender', fontdict={'fontsize' : 8, 'weight' : 'bold'})
plt.ylabel('Frequency', fontdict={'fontsize' : 8, 'weight' : 'bold'})
sns.despine()
plt.show()

In [ ]:
termination_dep = terminated_df['department_name'].value_counts().reset_index().head(8)

In [ ]:
# Creating a BarPlot to visualize the distribution of terminated employees By Department.
plt.figure(figsize=(8,4))
sns.barplot(x='department_name', y='count', data=termination_dep, color='#48cae4', edgecolor='#0096c7')
plt.title('Attrition Count By Department', fontdict={'fontsize' : 9, 'weight' : 'bold'})
plt.xticks(rotation=20)
plt.xlabel('Department', fontdict={'fontsize' : 8, 'weight' : 'bold'})
plt.ylabel('Frequency', fontdict={'fontsize' : 8, 'weight' : 'bold'})

sns.despine()
plt.show()

In [ ]:
termination_loc = terminated_df['city_name'].value_counts().reset_index().head(8)

In [ ]:
# Creating a BarPlot to visualize the distribution of terminated employees By City.
plt.figure(figsize=(8,4))
sns.barplot(x='city_name', y='count', data=termination_loc, color='#caf0f8', edgecolor='#0096c7')
plt.title('Attrition Count By City', fontdict={'fontsize' : 9, 'weight' : 'bold'})
plt.xticks(rotation=20)
plt.xlabel('City', fontdict={'fontsize' : 9, 'weight' : 'bold'})
plt.ylabel('Frequency', fontdict={'fontsize' : 8, 'weight' : 'bold'})

sns.despine()
plt.show()

In [ ]:
scaled_df = df.copy()

In [ ]:
# Converting the 'recorddate_key' and 'orighiredate_key' columns to datetime objects

scaled_df['recorddate_key'] = pd.to_datetime(scaled_df['recorddate_key'])
scaled_df['orighiredate_key'] = pd.to_datetime(scaled_df['orighiredate_key'])

In [ ]:
# Selecting columns with data type 'object' from the DataFrame.

obj_cols = scaled_df.select_dtypes(include=('object')).columns

In [ ]:
# Using LabelEncoder to encode categorical columns with 'object' data type in the DataFrame.

from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()

for col in obj_cols:
    scaled_df[col] = encoder.fit_transform(scaled_df[col])

In [ ]:
# Extracting year, month, and day components from datetime columns in the DataFrame.

scaled_df['recorddate_key_year'] = scaled_df['recorddate_key'].dt.year
scaled_df['recorddate_key_month'] = scaled_df['recorddate_key'].dt.month
scaled_df['recorddate_key_day'] = scaled_df['recorddate_key'].dt.day

scaled_df['orighiredate_key_year'] = scaled_df['orighiredate_key'].dt.year
scaled_df['orighiredate_key_month'] = scaled_df['orighiredate_key'].dt.month
scaled_df['orighiredate_key_day'] = scaled_df['orighiredate_key'].dt.day

In [ ]:
# Dropping the original datetime columns ('recorddate_key' and 'orighiredate_key') from the DataFrame.

scaled_df.drop(columns=['recorddate_key', 'orighiredate_key'], inplace=True)

In [ ]:
scaled_df

In [ ]:
# Scaling selected numerical columns in the DataFrame using StandardScaler.

cols = ['age', 'STATUS_YEAR', 'recorddate_key_year', 'orighiredate_key_year']

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

for col in cols:
    scaled_df[col] = scaler.fit_transform(scaled_df[[col]])

In [ ]:
# Splitting the DataFrame into features (x) and target variable (y).

x = scaled_df.drop(columns=['STATUS'])
y = scaled_df['STATUS']

In [ ]:
# Splitting the dataset into training and testing sets using train_test_split.

from sklearn.model_selection import train_test_split

xtrain, xtest, ytrain, ytest = train_test_split(x, y, test_size=0.2, random_state=4)

In [ ]:
# Printing the shapes of the training and testing sets to verify the data splitting.

print(f'Shape of xtrain : {xtrain.shape}')
print(f'Shape of xtest : {xtest.shape}')
print(f'Shape of ytrain : {ytrain.shape}')
print(f'Shape of ytest : {ytest.shape}')

In [ ]:
# Creating a Sequential model for binary classification.
Model = Sequential()

# Adding the Input layer with 32 units, ReLU activation, and input dimension of 14.
Model.add(Dense(units=32, activation='relu', input_dim=14))
Model.add(BatchNormalization())                                         # Adding BatchNormalization for improved training stability.

# Adding the Hidden layers with ReLU activation.
Model.add(Dense(units=64, activation='relu'))
Model.add(Dropout(0.3))

Model.add(Dense(units=128, activation='relu'))
Model.add(Dropout(0.3))

# Adding the output layer with 1 unit and sigmoid activation for binary classification.
Model.add(Dense(units=1, activation='sigmoid'))

# Compiling the model using the Adam optimizer and binary crossentropy loss.
# Monitoring accuracy as the evaluation metric.
Model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [ ]:
# Creating an EarlyStopping callback to halt training when accuracy stops improving.

early_stop = EarlyStopping(monitor='accuracy', patience=5, mode='auto', verbose=0, restore_best_weights=True)

In [ ]:
# Training the model on the training data with a specified number of epochs, batch size, and early stopping callback.

Model.fit(xtrain, ytrain, epochs=100, batch_size=64, callbacks=[early_stop])

In [ ]:
# Evaluate the model on the train set and retrieve the train accuracy

train_accuracy = Model.evaluate(xtrain, ytrain)
print(f'Training Accuracy : {round(train_accuracy[1] * 100, 2)}%')

In [ ]:
# Evaluate the model on the test set and retrieve the test accuracy

test_accuracy = Model.evaluate(xtest, ytest)
print(f'Testing Accuracy : {round(test_accuracy[1] * 100, 2)}%')

In [ ]:
# Prediction on Training set
train_pred_prob = Model.predict(xtrain)

# Convert the predictions to binary predictions (0 or 1) using a threshold
threshold = 0.5
train_pred_binary = (train_pred_prob > threshold).astype('int')

print("Classification Report for Training Set\n")
print(metrics.classification_report(ytrain, train_pred_binary))

In [ ]:
# Prediction on Testing set
test_pred_prob = Model.predict(xtest)

# Convert the predictions to binary predictions (0 or 1) using a threshold
threshold = 0.5
test_pred_binary = (test_pred_prob > threshold).astype('int')

print("Classification Report for Testing Set\n")
print(metrics.classification_report(ytest, test_pred_binary))

In [ ]:
# Creating a Confusion Matrix

cm_train = metrics.confusion_matrix(ytrain, train_pred_binary)
cm_test = metrics.confusion_matrix(ytest, test_pred_binary)

# Creating ConfusionMatrixDisplay object

cm_train_dis = metrics.ConfusionMatrixDisplay(cm_train, display_labels=['Active', 'Terminated'])
cm_test_dis = metrics.ConfusionMatrixDisplay(cm_test, display_labels=['Active', 'Terminated'])

fig, axs = plt.subplots(1, 2, figsize=(14, 4))

cm_train_dis.plot(ax=axs[0], values_format="d")
cm_test_dis.plot(ax=axs[1], values_format="d")

# Adding titles and labels
axs[0].set_title("Train Confusion Matrix")
axs[1].set_title("Test Confusion Matrix")

plt.show()

### **Testing**

In [ ]:
class_labels = ['Active', 'Terminated']

encoder.fit(class_labels)

In [ ]:
# Creating a test DataFrame by concatenating features (xtest) and target variable (ytest) along the columns.

test_df = pd.concat([xtest, ytest], axis=1)

In [ ]:
sample_record = test_df[test_df['STATUS'] == 1].sample(1)
sample_record

In [ ]:
sample = sample_record.drop(columns=['STATUS'])

In [ ]:
# Function for making predictions using the trained model.

def Predict(features):

    pred_prob = Model.predict(features)                       # Predicting probabilities using the trained model.

    threshold = 0.5
    pred_prob_binary = (pred_prob > threshold).astype('int')  # convert probabilities to binary predictions.

    pred_class = encoder.inverse_transform(pred_prob_binary)  # Inverse transforming binary predictions to obtain the predicted class.
    
    return pred_class

Predict(sample)